# LensWord — Text Preprocessing (Fixed)

This notebook prepares data for the LSTM model.

**Pipeline rules enforced:**
- Raw inputs are read-only — `amazon_reviews_cleaned.csv` is never overwritten
- Split happens BEFORE vocabulary is built
- Vocabulary fitted on training data only (no leakage)
- No SMOTE (invalid on token sequences — class-weighted loss used instead)
- Test texts saved with indices for provenance (fixes Error 6)
- Notebook must survive Restart & Run All from a fresh clone

In [1]:
# ── CELL 1: Import libraries ──
import pandas as pd
import numpy as np
import torch
from collections import Counter
import re
import sys
import os
import pickle
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

sys.path.append(os.path.abspath('..'))
from src.config import *

print("Libraries imported successfully!")
print(f"MAX_VOCAB_SIZE: {MAX_VOCAB_SIZE}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")
print(f"BATCH_SIZE:     {BATCH_SIZE}")

Libraries imported successfully!
MAX_VOCAB_SIZE: 4340
MAX_SEQ_LENGTH: 50
BATCH_SIZE:     32


In [2]:
# ── CELL 2: Load Amazon data — READ ONLY ──
# notebook 01 (EDA) already mapped star ratings to sentiment labels
# amazon_reviews_cleaned.csv is the output of notebook 01 — never overwrite it

df_amazon = pd.read_csv('../data/amazon_reviews_cleaned.csv')

print(f"Amazon reviews loaded: {len(df_amazon)}")
print("\nDistribution:")
print(df_amazon['sentiment'].value_counts())
print("\nSample:")
print(df_amazon.head(3))

Amazon reviews loaded: 3149

Distribution:
sentiment
Positive    2741
Negative     256
Neutral      152
Name: count, dtype: int64

Sample:
                                    verified_reviews sentiment
0                                      Love my Echo!  Positive
1                                          Loved it!  Positive
2  Sometimes while playing a game, you can answer...  Positive


In [3]:
# ── CELL 3: Load and sample Yelp data from HuggingFace ──
# We sample more Negative and Neutral to compensate for Amazon's Positive skew
from datasets import load_dataset

print("Loading Yelp dataset from HuggingFace...")
dataset = load_dataset("Yelp/yelp_review_full", split="train")
df_hf = dataset.to_pandas()
df_hf = df_hf.rename(columns={'text': 'verified_reviews'})

def map_yelp_sentiment(label):
    if label <= 1:
        return 'Negative'
    elif label == 2:
        return 'Neutral'
    else:
        return 'Positive'

df_hf['sentiment'] = df_hf['label'].apply(map_yelp_sentiment)

# Filter to short reviews only (<=50 words) to match Amazon style
df_hf['word_count'] = df_hf['verified_reviews'].apply(lambda x: len(str(x).split()))
df_hf_short = df_hf[df_hf['word_count'] <= 50]

print(f"Short Yelp reviews available: {len(df_hf_short)}")
print(df_hf_short['sentiment'].value_counts())

# Sample more Negative and Neutral to compensate for Amazon's Positive skew
# Amazon: 2741 Positive, 256 Negative, 152 Neutral
# We add extra Negative and Neutral to balance the combined dataset
df_yelp_sampled = pd.concat([
    df_hf_short[df_hf_short['sentiment'] == 'Negative'].sample(3500, random_state=42),
    df_hf_short[df_hf_short['sentiment'] == 'Neutral'].sample(3500, random_state=42),
    df_hf_short[df_hf_short['sentiment'] == 'Positive'].sample(1000, random_state=42)
])
df_yelp_sampled = df_yelp_sampled[['verified_reviews', 'sentiment']].reset_index(drop=True)

print(f"\nSampled Yelp: {len(df_yelp_sampled)}")
print(df_yelp_sampled['sentiment'].value_counts())

Loading Yelp dataset from HuggingFace...


Short Yelp reviews available: 154809
sentiment
Positive    77095
Negative    51198
Neutral     26516
Name: count, dtype: int64

Sampled Yelp: 8000
sentiment
Negative    3500
Neutral     3500
Positive    1000
Name: count, dtype: int64


In [4]:
# ── CELL 4: Combine, deduplicate, save to NEW file ──
# Save to amazon_yelp_combined.csv — NOT amazon_reviews_cleaned.csv
# amazon_reviews_cleaned.csv is read-only (written only by notebook 01)

df_combined = pd.concat([df_amazon, df_yelp_sampled], ignore_index=True)

# Deduplicate before splitting — prevents train/test leakage
df_combined = df_combined.drop_duplicates(subset=['verified_reviews'])
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Combined total (after dedup): {len(df_combined)}")
print("\nDistribution:")
print(df_combined['sentiment'].value_counts())
print("\nPercentages:")
print((df_combined['sentiment'].value_counts(normalize=True) * 100).round(2))

# Save to NEW output file — inputs are never overwritten
df_combined.to_csv('../data/amazon_yelp_combined.csv', index=False)
print("\nSaved to: data/amazon_yelp_combined.csv")

# Use df_combined for all downstream steps
df = df_combined.copy()

Combined total (after dedup): 10299

Distribution:
sentiment
Negative    3705
Neutral     3604
Positive    2990
Name: count, dtype: int64

Percentages:
sentiment
Negative    35.97
Neutral     34.99
Positive    29.03
Name: proportion, dtype: float64

Saved to: data/amazon_yelp_combined.csv


In [5]:
# ── CELL 5: Map labels and clean text ──
label_map = {'Positive': 2, 'Neutral': 1, 'Negative': 0}
df['label'] = df['sentiment'].map(label_map)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_review'] = df['verified_reviews'].apply(clean_text)

# Remove reviews that clean to empty string
df = df[df['cleaned_review'].str.strip() != ''].reset_index(drop=True)

print(f"Reviews after cleaning: {len(df)}")
print("\nBefore:", df['verified_reviews'][0])
print("After: ", df['cleaned_review'][0])

Reviews after cleaning: 10294

Before: Great view of the strip and the Bellagio fountains.
After:  great view of the strip and the bellagio fountains


In [6]:
# ── CELL 6: Helper functions for tokenization and padding ──
def text_to_sequence(text, word2idx):
    words = text.split()
    return [word2idx.get(word, word2idx['<UNK>']) for word in words]

def pad_sequence_fn(sequence, max_length):
    if len(sequence) < max_length:
        sequence = sequence + [0] * (max_length - len(sequence))
    else:
        sequence = sequence[:max_length]
    return sequence

print("Helper functions defined.")

Helper functions defined.


In [7]:
# ── CELL 7: SPLIT FIRST (before vocabulary is built) ──
# Correct order: split -> fit vocab on train -> transform all
# This prevents vocabulary leakage into val/test sets

X_text = df['cleaned_review'].values
y = np.array(df['label'].tolist())
indices = np.arange(len(df))

# First split: 75% train, 25% temp
X_text_train, X_text_temp, y_train, y_temp, idx_train, idx_temp = train_test_split(
    list(X_text), y, indices,
    test_size=0.25, random_state=42, stratify=y)

# Second split: 60% val, 40% test from the 25% temp
X_text_val, X_text_test, y_val, y_test, idx_val, idx_test = train_test_split(
    X_text_temp, y_temp, idx_temp,
    test_size=0.4, random_state=42, stratify=y_temp)

print(f"Training:   {len(X_text_train)} reviews")
print(f"Validation: {len(X_text_val)} reviews")
print(f"Test:       {len(X_text_test)} reviews")

# Verify no overlap between splits
train_set = set(idx_train)
val_set   = set(idx_val)
test_set  = set(idx_test)
assert len(train_set & test_set) == 0, "Train/test overlap detected!"
assert len(train_set & val_set)  == 0, "Train/val overlap detected!"
assert len(val_set & test_set)   == 0, "Val/test overlap detected!"
print("\nNo train/val/test overlap — confirmed!")

Training:   7720 reviews
Validation: 1544 reviews
Test:       1030 reviews

No train/val/test overlap — confirmed!


In [8]:
# ── CELL 8: Build vocabulary on TRAINING DATA ONLY ──
# Fitting on all data (including test) is preprocessing leakage
# Correct: fit on train partition, transform all partitions

all_words = []
for review in X_text_train:
    all_words.extend(review.split())

word_counts = Counter(all_words)
print(f"Unique words in training data: {len(word_counts)}")

vocab = ['<PAD>', '<UNK>'] + [word for word, count in word_counts.most_common(MAX_VOCAB_SIZE - 2)]
word2idx = {word: idx for idx, word in enumerate(vocab)}

print(f"Vocabulary size: {len(vocab)}")
print(f"Sample entries:  {list(word2idx.items())[:5]}")

Unique words in training data: 14163
Vocabulary size: 4340
Sample entries:  [('<PAD>', 0), ('<UNK>', 1), ('the', 2), ('and', 3), ('i', 4)]


In [9]:
# ── CELL 9: Convert all splits to sequences using training vocabulary ──
X_train_seq = [pad_sequence_fn(text_to_sequence(t, word2idx), MAX_SEQ_LENGTH) for t in X_text_train]
X_val_seq   = [pad_sequence_fn(text_to_sequence(t, word2idx), MAX_SEQ_LENGTH) for t in X_text_val]
X_test_seq  = [pad_sequence_fn(text_to_sequence(t, word2idx), MAX_SEQ_LENGTH) for t in X_text_test]

X_train = np.array(X_train_seq)
X_val   = np.array(X_val_seq)
X_test  = np.array(X_test_seq)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (7720, 50)
X_val shape:   (1544, 50)
X_test shape:  (1030, 50)


In [10]:
# ── CELL 10: Save test texts for provenance ──
# This fixes Error 6 from the advisor review
# We persist the test rows' texts at split time so notebook 04 can
# join predictions back to the correct review texts using original_index

df_test_texts = pd.DataFrame({
    'original_index':  idx_test,
    'verified_reviews': [df['verified_reviews'].values[i] for i in idx_test],
    'cleaned_review':  X_text_test,
    'true_label':      y_test,
    'true_sentiment':  [['Negative', 'Neutral', 'Positive'][l] for l in y_test]
})
df_test_texts.to_csv('../data/test_texts.csv', index=False)
print(f"Test texts saved: {len(df_test_texts)} rows -> data/test_texts.csv")
print(df_test_texts.head(3))

Test texts saved: 1030 rows -> data/test_texts.csv
   original_index                                   verified_reviews  \
0            5048  Fast delivery and decent asian food in the mad...   
1            7254  This course is A-OK! Paid 20$ for cart with a ...   
2            9864  Is this place still open??? drove by a cpl wee...   

                                      cleaned_review  true_label  \
0  fast delivery and decent asian food in the mad...           1   
1  this course is aok paid for cart with a am tee...           1   
2  is this place still open drove by a cpl weeks ...           1   

  true_sentiment  
0        Neutral  
1        Neutral  
2        Neutral  


In [11]:
# ── CELL 11: Convert to PyTorch tensors and save ──
X_train_tensor = torch.tensor(X_train, dtype=torch.long)
X_val_tensor   = torch.tensor(X_val,   dtype=torch.long)
X_test_tensor  = torch.tensor(X_test,  dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_val_tensor   = torch.tensor(y_val,   dtype=torch.long)
y_test_tensor  = torch.tensor(y_test,  dtype=torch.long)

# Save tensors (no SMOTE tensors — SMOTE is invalid on token sequences)
torch.save(X_train_tensor, '../data/X_train.pt')
torch.save(X_val_tensor,   '../data/X_val.pt')
torch.save(X_test_tensor,  '../data/X_test.pt')
torch.save(y_train_tensor, '../data/y_train.pt')
torch.save(y_val_tensor,   '../data/y_val.pt')
torch.save(y_test_tensor,  '../data/y_test.pt')

# Save vocabulary (fitted on training data only)
with open('../data/word2idx.pkl', 'wb') as f:
    pickle.dump(word2idx, f)

print("All tensors and vocabulary saved!")
print(f"X_train: {X_train_tensor.shape}")
print(f"X_val:   {X_val_tensor.shape}")
print(f"X_test:  {X_test_tensor.shape}")

All tensors and vocabulary saved!
X_train: torch.Size([7720, 50])
X_val:   torch.Size([1544, 50])
X_test:  torch.Size([1030, 50])


In [12]:
# ── CELL 12: Final summary ──
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_tensor,   y_val_tensor),   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test_tensor,  y_test_tensor),  batch_size=BATCH_SIZE, shuffle=False)

print("=" * 52)
print("PREPROCESSING SUMMARY — LensWord (Fixed)")
print("=" * 52)
print(f"Source files:        amazon_reviews_cleaned.csv (read-only)")
print(f"                     Yelp/yelp_review_full (HuggingFace)")
print(f"Output file:         amazon_yelp_combined.csv")
print(f"Total reviews:       {len(df)}")
print(f"Vocabulary size:     {len(vocab)} (fitted on training only)")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")
print(f"Training set:        {X_train_tensor.shape[0]}")
print(f"Validation set:      {X_val_tensor.shape[0]}")
print(f"Test set:            {X_test_tensor.shape[0]}")
print(f"Training batches:    {len(train_loader)}")
print(f"Batch size:          {BATCH_SIZE}")
print(f"SMOTE applied:       No (invalid on token sequences)")
print(f"Imbalance handling:  Class-weighted loss in notebook 03")
print(f"Test texts saved:    data/test_texts.csv (for error analysis)")
print("=" * 52)

# Class distribution per split
label_names = ['Negative', 'Neutral', 'Positive']
for name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    unique, counts = np.unique(y_split, return_counts=True)
    dist = {label_names[u]: int(c) for u, c in zip(unique, counts)}
    print(f"{name}: {dist}")

PREPROCESSING SUMMARY — LensWord (Fixed)
Source files:        amazon_reviews_cleaned.csv (read-only)
                     Yelp/yelp_review_full (HuggingFace)
Output file:         amazon_yelp_combined.csv
Total reviews:       10294
Vocabulary size:     4340 (fitted on training only)
Max sequence length: 50
Training set:        7720
Validation set:      1544
Test set:            1030
Training batches:    242
Batch size:          32
SMOTE applied:       No (invalid on token sequences)
Imbalance handling:  Class-weighted loss in notebook 03
Test texts saved:    data/test_texts.csv (for error analysis)
Train: {'Negative': 2778, 'Neutral': 2703, 'Positive': 2239}
Val: {'Negative': 556, 'Neutral': 540, 'Positive': 448}
Test: {'Negative': 371, 'Neutral': 361, 'Positive': 298}
